# 🔍 Explorador Simplificado de Contagem de Multidão RGBT (Single Image Pair)

Este notebook foi criado para **eliminar a complexidade das esteiras de dados e abstrações CLI**, permitindo a exploração direta, clara e passo a passo da contagem de pessoas com **um único par de imagens (Óptica RGB + Térmica)**.

---

## 🛠 Passo 1: Importar Bibliotecas Essenciais e Configurar o Caminho

In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Adiciona o diretório app/ ao path do Python para importar os módulos nativos
app_dir = Path('../').resolve()
if str(app_dir) not in sys.path:
    sys.path.insert(0, str(app_dir))

from head_counting.config import PipelineConfig
from head_counting.preprocessing import RGBTImageEqualizer
from head_counting.model import DEFModelHandler

print(f'✅ Módulos importados com sucesso a partir de: {app_dir}')

## 📸 Passo 2: Carregar as Imagens Brutas de Entrada (Landing / Bronze)

Carregamos a imagem óptica Wide (`DJI_0789_W.JPG`) e a imagem térmica (`DJI_0790_T.JPG`).

In [ ]:
# Caminhos das mídias brutas da câmera DJI
rgb_path = app_dir / 'data/landing/DJI_0789_W.JPG'
thermal_path = app_dir / 'data/landing/DJI_0790_T.JPG'

# Carregar via OpenCV (formato BGR)
img_rgb_raw = cv2.imread(str(rgb_path))
img_th_raw = cv2.imread(str(thermal_path))

print(f'Resolução Nativa RGB (Wide 24mm): {img_rgb_raw.shape[1]}x{img_rgb_raw.shape[0]} px')
print(f'Resolução Nativa Térmica (40mm):  {img_th_raw.shape[1]}x{img_th_raw.shape[0]} px')

# Visualização lado a lado em RGB
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(img_rgb_raw, cv2.COLOR_BGR2RGB))
axes[0].set_title('1. Sensor Óptico RGB Bruto (Wide 24mm)')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_th_raw, cv2.COLOR_BGR2RGB))
axes[1].set_title('2. Sensor Térmico Bruto (40mm)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## ⚙️ Passo 3: Pré-Processamento (Alinhamento de FOV + Correção de Lente + Equalização)

Instanciamos o `RGBTImageEqualizer` diretamente para:
1. **Desditorcer a lente Wide** de 24mm (`undistort_lens=True`).
2. **Realizar o corte de FOV 70%** combinando a proporção de aspecto **5:4** nativa do sensor térmico.
3. **Aplicar a translação fina de alinhamento** (`shift_rgb_x=-22`, `shift_rgb_y=-23`).
4. **Equalizar a imagem térmica** com CLAHE para maximizar o contraste de calor.

In [ ]:
# Instancia o pré-processador
preprocessor = RGBTImageEqualizer(
    target_size=(1280, 1024),
    fov_crop_ratio=0.70,
    undistort_lens=True,
    shift_rgb_x=-22,
    shift_rgb_y=-23,
    thermal_clahe=True
)

# Processa e alinha o par de matrizes
img_rgb_aligned, img_th_aligned = preprocessor.process_pair(img_rgb_raw, img_th_raw)

# Cria a sobreposição 50% RGB + 50% Térmica para auditoria visual
blend_overlay = preprocessor.create_blend_overlay(img_rgb_aligned, img_th_aligned, alpha=0.5)

# Visualização do Alinhamento Gerado
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(blend_overlay, cv2.COLOR_BGR2RGB))
plt.title('3. Overlay de Auditoria de Alinhamento (50% RGB + 50% Térmica)')
plt.axis('off')
plt.show()

## 🧠 Passo 4: Carregar o Modelo e Executar a Inferência Multimodal

Instanciamos o `DEFModelHandler` utilizando as configurações e executamos a predição no par de imagens igualadas.

In [ ]:
# Carrega as configurações do pipeline
config = PipelineConfig.from_yaml(app_dir / 'data_rgbt_images.yaml')

# Inicializa o Handler do Modelo
model_handler = DEFModelHandler(config)
model_handler.setup_runtime()
model_handler.load_model()

# Executa a inferência multimodal no par de matrizes alinhadas
results = model_handler.predict_batch([img_rgb_aligned], [img_th_aligned])

count_estimate = results[0]['count']
density_map = results[0]['density_map']

print('=' * 55)
print(f'🎉 CONTAGEM ESTIMADA NA IMAGEM: {count_estimate:.2f} pessoas')
print('=' * 55)

## 📊 Passo 5: Painel Visual Completo de Resultados

Exibimos o quadrante contendo: RGB Alinhada, Térmica Equalizada, Overlay Blend e o Mapa de Densidade (*Heatmap*).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Imagem RGB Alinhada
axes[0, 0].imshow(cv2.cvtColor(img_rgb_aligned, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('1. Imagem RGB Alinhada (1280x1024)')
axes[0, 0].axis('off')

# 2. Imagem Térmica Equalizada
axes[0, 1].imshow(cv2.cvtColor(img_th_aligned, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title('2. Imagem Térmica CLAHE (1280x1024)')
axes[0, 1].axis('off')

# 3. Overlay Blend
axes[1, 0].imshow(cv2.cvtColor(blend_overlay, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title('3. Sobreposição de Co-Registro (Blend 50/50)')
axes[1, 0].axis('off')

# 4. Mapa de Densidade Heatmap
im = axes[1, 1].imshow(density_map, cmap='jet')
axes[1, 1].set_title(f'4. Mapa de Densidade Estimado ({count_estimate:.1f} pessoas)')
axes[1, 1].axis('off')
fig.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

plt.suptitle('Painel Completo de Inferência Multimodal RGBT', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()